In [3]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import os
from scipy.signal import butter, filtfilt
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import LeaveOneGroupOut
import warnings
warnings.filterwarnings('ignore')

# ===== 3クラスに変換 =====
def to_3class(enjoyment):
    if enjoyment <= 2: return 0   # 低
    elif enjoyment == 3: return 1  # 中
    else: return 2                 # 高

# ===== バンドパスフィルタ =====
def bandpass(data, lo, hi, sf=250):
    nyq = sf / 2
    b, a = butter(4, [lo/nyq, hi/nyq], btype='band')
    return filtfilt(b, a, data, axis=-1)

# ===== EEGNet（軽量版）=====
class EEGNetBranch(nn.Module):
    def __init__(self, n_channels=4, n_times=7500, out_dim=64):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(1, 8, (1, 64), padding=(0, 32), bias=False),
            nn.BatchNorm2d(8),
            nn.Conv2d(8, 16, (n_channels, 1), groups=8, bias=False),
            nn.BatchNorm2d(16),
            nn.ELU(),
            nn.AvgPool2d((1, 4)),
            nn.Dropout(0.5),
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(16, 16, (1, 16), padding=(0, 8), groups=16, bias=False),
            nn.Conv2d(16, 32, (1, 1), bias=False),
            nn.BatchNorm2d(32),
            nn.ELU(),
            nn.AvgPool2d((1, 8)),
            nn.Dropout(0.5),
        )
        fc_in = 32 * (n_times // 32)
        self.fc = nn.Linear(fc_in, out_dim)

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = x.view(x.size(0), -1)
        return torch.relu(self.fc(x))

# ===== マルチブランチモデル =====
class MultiBranchEEG(nn.Module):
    def __init__(self, n_classes=3, n_channels=4, n_times=7500):
        super().__init__()
        self.theta_branch = EEGNetBranch(n_channels, n_times, 64)
        self.alpha_branch = EEGNetBranch(n_channels, n_times, 64)
        self.beta_branch  = EEGNetBranch(n_channels, n_times, 64)

        self.classifier = nn.Sequential(
            nn.Linear(64 * 3, 128),
            nn.ELU(),
            nn.Dropout(0.5),
            nn.Linear(128, n_classes),
        )

    def forward(self, x_theta, x_alpha, x_beta):
        f_theta = self.theta_branch(x_theta)
        f_alpha = self.alpha_branch(x_alpha)
        f_beta  = self.beta_branch(x_beta)
        fused = torch.cat([f_theta, f_alpha, f_beta], dim=1)
        return self.classifier(fused)

# ===== データセット =====
class MultiBranchDataset(Dataset):
    def __init__(self, ts_dir, labels_df, n_classes=3):
        self.theta_list, self.alpha_list, self.beta_list = [], [], []
        self.labels = []

        for _, row in labels_df.iterrows():
            fname = f"sub{int(row['subject']):03d}_ses{int(row['song_id']):02d}.npy"
            fpath = os.path.join(ts_dir, fname)
            if not os.path.exists(fpath):
                continue

            data = np.load(fpath).astype(np.float32)  # (4, 7500)

            def norm(x):
                return (x - x.mean()) / (x.std() + 1e-8)

            theta = norm(bandpass(data, 4, 8).astype(np.float32))
            alpha = norm(bandpass(data, 8, 13).astype(np.float32))
            beta  = norm(bandpass(data, 13, 30).astype(np.float32))

            self.theta_list.append(theta)
            self.alpha_list.append(alpha)
            self.beta_list.append(beta)

            if n_classes == 3:
                self.labels.append(to_3class(int(row['enjoyment'])))
            else:
                self.labels.append(int(row['enjoyment']) - 1)

    def __len__(self): return len(self.labels)

    def __getitem__(self, idx):
        def to_tensor(x):
            return torch.tensor(x).unsqueeze(0)  # (1, 4, 7500)
        return (to_tensor(self.theta_list[idx]),
                to_tensor(self.alpha_list[idx]),
                to_tensor(self.beta_list[idx]),
                torch.tensor(self.labels[idx], dtype=torch.long))

# ===== メイン =====
def run():
    data_root = r"C:\Users\hiro2\Documents\EEG_project\data"
    ts_dir    = os.path.join(data_root, "timeseries")
    labels_df = pd.read_csv(os.path.join(data_root, "features_clean.csv"))
    device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"[MultiBranch] device: {device}")

    dataset = MultiBranchDataset(ts_dir, labels_df, n_classes=3)
    print(f"[MultiBranch] データ数: {len(dataset)}")

    label_counts = pd.Series([dataset[i][3].item() for i in range(len(dataset))]).value_counts().sort_index()
    print(f"[MultiBranch] ラベル分布: {label_counts.to_dict()}")

    groups   = labels_df['subject'].values[:len(dataset)]
    logo     = LeaveOneGroupOut()
    all_accs = []

    for fold, (tr_idx, te_idx) in enumerate(
            logo.split(range(len(dataset)),
                       [dataset[i][3].item() for i in range(len(dataset))],
                       groups)):

        tr_loader = DataLoader(
            torch.utils.data.Subset(dataset, tr_idx),
            batch_size=8, shuffle=True)
        te_loader = DataLoader(
            torch.utils.data.Subset(dataset, te_idx),
            batch_size=8)

        model     = MultiBranchEEG(n_classes=3, n_channels=4,
                                   n_times=7500).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3,
                                     weight_decay=1e-4)
        counts = np.array([112, 69, 57], dtype=np.float32)
        weights = torch.tensor(1.0 / counts).to(device)
        weights = weights / weights.sum()
        criterion = nn.CrossEntropyLoss(weight=weights)
        scheduler = torch.optim.lr_scheduler.StepLR(
            optimizer, step_size=20, gamma=0.5)

        for epoch in range(60):
            model.train()
            for batch in tr_loader:
                xt, xa, xb, y = [b.to(device) for b in batch]
                optimizer.zero_grad()
                loss = criterion(model(xt, xa, xb), y)
                loss.backward()
                optimizer.step()
            scheduler.step()

        model.eval()
        correct = 0
        with torch.no_grad():
            for batch in te_loader:
                xt, xa, xb, y = [b.to(device) for b in batch]
                correct += (model(xt, xa, xb).argmax(1) == y).sum().item()
        acc = correct / len(te_idx)
        all_accs.append(acc)
        print(f"[MultiBranch] Fold {fold+1:2d}: acc={acc:.3f}")

    chance = 1/3
    print(f"\n[MultiBranch] 平均acc: {np.mean(all_accs):.3f} ± {np.std(all_accs):.3f}")
    print(f"[MultiBranch] チャンス精度: {chance:.3f} (3クラス)")

if __name__ == '__main__':
    run()

[MultiBranch] device: cpu
[MultiBranch] データ数: 238
[MultiBranch] ラベル分布: {0: 112, 1: 69, 2: 57}


KeyboardInterrupt: 

In [4]:
pip show muselsl

Note: you may need to restart the kernel to use updated packages.


In [5]:
pip install muselsl pylsl


   ---------------------------------------- 0/8 [pyserial]
   ---------- ----------------------------- 2/8 [bitarray]
   ---------- ----------------------------- 2/8 [bitarray]
   -------------------- ------------------- 4/8 [pylsl]
   ------------------------- -------------- 5/8 [pygatt]
   ------------------------------ --------- 6/8 [bitstring]
   ---------------------------------------- 8/8 [muselsl]

Note: you may need to restart the kernel to use updated packages.
